# Lab 8.4: RAG in Practice - Modern Frameworks & Tools

**Course:** Advanced Natural Language Processing (NLP702/806)

**Instructor:** Dr. Fajri Koto

---

In Labs 8.1-8.3 we built RAG from scratch. In practice, engineers use **frameworks** and **vector databases** to build production RAG systems faster and more reliably.

### What We'll Cover:

1. **Vector stores** - Storing and searching embeddings with ChromaDB
2. **LangChain** - The most popular RAG orchestration framework
3. **LlamaIndex** - A retrieval-focused alternative
4. **Agentic RAG** - The frontier: LLM agents that plan their own retrieval

In [1]:
from datasets import load_dataset

# Load our familiar dataset
corpus_dataset = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus", split="passages")
qa_dataset = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer", split="test")

passages = [item["passage"] for item in corpus_dataset]
print(f"Loaded {len(passages)} passages, {len(qa_dataset)} QA pairs")

/Users/hasaniqbal/Documents/! PhD - MBZUAI/TA/NLP701:806/Lab 8/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 3200 passages, 918 QA pairs


## Section 1: Vector Stores with ChromaDB

In Labs 8.2 and 8.3 we computed embeddings and stored them in a NumPy array. This works for small datasets, but **vector databases** are essential for production:

| Feature | NumPy Array | Vector Database (e.g., ChromaDB) |
|---------|------------|----------------------------------|
| Persistence | In-memory only | Disk-backed |
| Metadata filtering | Manual | Built-in |
| Scalability | ~100K vectors | Millions+ |
| Updates | Rebuild index | Add/delete individual docs |

**ChromaDB** is open-source, runs locally, and is the easiest to get started with.

In [2]:
import chromadb

# Create an in-memory ChromaDB client (use PersistentClient for disk storage)
chroma_client = chromadb.Client()

# Create a collection - ChromaDB handles embedding automatically
collection = chroma_client.create_collection(
    name="wiki_passages",
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity
)

# Add passages (ChromaDB uses its own default embedding model)
# We use a subset to keep things fast
subset_size = 2000
collection.add(
    documents=passages[:subset_size],
    ids=[f"passage_{i}" for i in range(subset_size)],
    metadatas=[{"source": "wikipedia", "index": i} for i in range(subset_size)]
)

print(f"Collection '{collection.name}' has {collection.count()} documents")

Collection 'wiki_passages' has 2000 documents


In [3]:
# Query the collection - embedding + search in one call
query = "What is the speed of light?"

results = collection.query(
    query_texts=[query],
    n_results=3,
)

print(f"Query: {query}\n")
for i, (doc, distance) in enumerate(zip(results["documents"][0], results["distances"][0])):
    print(f"[{i+1}] (distance={distance:.4f})")
    print(f"    {doc[:150]}...\n")

Query: What is the speed of light?

[1] (distance=0.6761)
    Newton argued that light is composed of particles or corpuscles and were refracted by accelerating toward the denser medium, but he had to associate t...

[2] (distance=0.6890)
    150px...

[3] (distance=0.6933)
    # Newton's Second Law states that an applied force,   F  , on an object equals the time rate of change of its momentum,   p  . Mathematically, this is...



In [4]:
# Metadata filtering - a key advantage of vector databases
# For example, filter by source or any custom metadata
results_filtered = collection.query(
    query_texts=["theory of relativity"],
    n_results=3,
    where={"source": "wikipedia"}  # Only search Wikipedia passages
)

print("Filtered query results:")
for i, doc in enumerate(results_filtered["documents"][0]):
    print(f"  [{i+1}] {doc[:120]}...")

Filtered query results:
  [1] * Richard de Villamil. Newton, The man. G.D. Knox, London, 1931. Preface by Albert Einstein. Reprinted by Johnson Reprin...
  [2] The famous three laws of motion:...
  [3] In mechanics, Newton enunciated the principles of conservation of momentum and angular momentum. In optics, he invented ...


### Other Popular Vector Stores

| Vector Store | Type | Best For |
|-------------|------|----------|
| **ChromaDB** | Embedded / local | Prototyping, small-medium scale |
| **FAISS** (Meta) | Library | High-performance similarity search |
| **Pinecone** | Managed cloud | Production, zero-ops |
| **Weaviate** | Self-hosted / cloud | Hybrid search (vector + keyword) |
| **Qdrant** | Self-hosted / cloud | Filtering + vector search |
| **Milvus** | Self-hosted | Large-scale (billions of vectors) |

## Section 2: RAG with LangChain

[LangChain](https://python.langchain.com/) is the most widely adopted framework for building LLM applications. It provides modular components that snap together:

```
Document Loader --> Text Splitter --> Embeddings --> Vector Store --> Retriever --> LLM Chain
```

Each component is swappable - change your vector store, embedding model, or LLM without rewriting the pipeline.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [7]:
# Step 1: Wrap passages as LangChain Documents
docs = [
    Document(page_content=p, metadata={"source": "wikipedia", "index": i})
    for i, p in enumerate(passages[:subset_size])
]

# Step 2: Split into chunks (LangChain's RecursiveCharacterTextSplitter
# is smarter than fixed-size - it tries paragraph > sentence > word boundaries)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # characters, not words
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)
split_docs = text_splitter.split_documents(docs)
print(f"{len(docs)} documents --> {len(split_docs)} chunks")
print(f"\nExample chunk ({len(split_docs[0].page_content)} chars):")
print(split_docs[0].page_content[:200])

2000 documents --> 2867 chunks

Example chunk (250 chars):
Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in


In [8]:
# Step 3: Create vector store with HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    collection_name="langchain_wiki"
)

print(f"Vector store created with {vectorstore._collection.count()} chunks")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 22203.49it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store created with 2867 chunks


In [9]:
# Step 4: Build the RAG chain using LCEL (LangChain Expression Language)
# This is the modern way to compose LangChain components
llm = ChatOllama(model="llama3.2:3b", temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_prompt = ChatPromptTemplate.from_template(
    """Answer the question based ONLY on the provided context. If the context \
does not contain enough information, say "I cannot answer this based on the provided context."

Context:
{context}

Question: {question}

Answer:"""
)

# LCEL chain: retriever -> format -> prompt -> LLM -> parse
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("LangChain RAG pipeline ready!")

LangChain RAG pipeline ready!


In [10]:
# Test the LangChain RAG pipeline
test_questions = [
    "What is the speed of light?",
    "Who wrote Romeo and Juliet?",
    "When was the United Nations founded?",
]

for q in test_questions:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()

Q: What is the speed of light?
A: I cannot answer this based on the provided context. The context discusses the nature of light and how it was understood by different physicists, but it does not mention the speed of light.

Q: Who wrote Romeo and Juliet?
A: I cannot answer this based on the provided context. The text does not mention Romeo and Juliet at all. It appears to be discussing historical figures, their relationships, and a biography website, but it does not contain any information about Shakespeare's works or Romeo and Juliet specifically.

Q: When was the United Nations founded?
A: I cannot answer this based on the provided context. The context only mentions Indonesia's relations with the UN, its participation in various international organizations, and historical events related to its independence and colonial history, but does not provide information about the founding of the United Nations itself.



### What LangChain Gave Us

We used **LCEL (LangChain Expression Language)** — the modern way to compose LangChain components using the `|` (pipe) operator:

```python
chain = {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | parser
```

Compare with Lab 8.2 where we manually wrote `build_rag_prompt()` and `generate_answer()`. LangChain handles:
- Retrieval and document formatting
- Prompt template construction
- LLM invocation and output parsing

**Trade-off:** More abstraction = faster development, but less control over the exact prompt and retrieval logic.

## Section 3: RAG with LlamaIndex

[LlamaIndex](https://docs.llamaindex.ai/) is built specifically for connecting LLMs to data. Where LangChain is a general-purpose LLM framework, LlamaIndex focuses on **indexing and retrieval**.

Key difference: LlamaIndex provides higher-level abstractions for document ingestion and query engines.

In [11]:
from llama_index.core import VectorStoreIndex, Document as LIDocument, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama as LIOllama

In [12]:
# Configure LlamaIndex to use local models
Settings.llm = LIOllama(model="llama3.2:3b", request_timeout=120.0)
Settings.embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")
Settings.chunk_size = 512
Settings.chunk_overlap = 100

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8210.53it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
# Create LlamaIndex documents and build the index
# LlamaIndex handles chunking automatically based on Settings
li_docs = [
    LIDocument(text=p, metadata={"source": "wikipedia", "index": i})
    for i, p in enumerate(passages[:subset_size])
]

index = VectorStoreIndex.from_documents(li_docs, show_progress=True)
print("LlamaIndex index built!")

Generating embeddings: 100%|██████████| 2001/2001 [00:05<00:00, 395.50it/s]


LlamaIndex index built!


In [14]:
# Create a query engine - LlamaIndex's equivalent of a RAG chain
query_engine = index.as_query_engine(similarity_top_k=3)

# Test it
for q in test_questions:
    response = query_engine.query(q)
    print(f"Q: {q}")
    print(f"A: {response}")
    print(f"   Sources: {len(response.source_nodes)} nodes retrieved")
    print()

Q: What is the speed of light?
A: A fundamental constant in physics. It's a number that has been extensively studied and measured over the years, with many different experiments contributing to our understanding of its value.
   Sources: 3 nodes retrieved

Q: Who wrote Romeo and Juliet?
A: William Shakespeare.
   Sources: 3 nodes retrieved

Q: When was the United Nations founded?
A: The UN was founded on October 24, 1945.
   Sources: 3 nodes retrieved



### LangChain vs LlamaIndex

| Aspect | LangChain | LlamaIndex |
|--------|-----------|------------|
| **Focus** | General LLM orchestration | Data indexing & retrieval |
| **RAG setup** | More manual wiring | More automatic (handles chunking, indexing) |
| **Flexibility** | High - swap any component | High for retrieval, less for non-RAG tasks |
| **Learning curve** | Steeper (many abstractions) | Gentler for RAG-specific tasks |
| **Ecosystem** | Larger community, more integrations | Strong retrieval-specific features |
| **Production** | LangGraph for complex workflows | Workflows for multi-step agents |

**In practice**: Many teams use both - LlamaIndex for ingestion/indexing, LangChain for orchestration.

## Section 4: Agentic RAG - The Frontier

Standard RAG follows a fixed pipeline: **retrieve -> generate**. But what if the first retrieval doesn't find good results? What if the question needs multiple searches?

**Agentic RAG** gives the LLM agency over the retrieval process:

```
Standard RAG:    Question --> Retrieve --> Generate --> Answer

Agentic RAG:     Question --> Agent decides:
                                |- Search knowledge base?
                                |- Reformulate query and search again?
                                |- Search a different source?
                                |- Enough info? --> Generate answer
                                |- Answer incomplete? --> Search more
```

The agent can **plan**, **reflect**, and **iterate** - much like how a human researcher would approach a question.

In [15]:
from openai import OpenAI
import json

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "llama3.2:3b"


def search_knowledge_base(query, top_k=3):
    """Search the ChromaDB collection."""
    results = collection.query(query_texts=[query], n_results=top_k)
    return results["documents"][0]


def agentic_rag(question, max_iterations=3, verbose=True):
    """
    Agentic RAG: the LLM decides whether to search, refine, or answer.
    
    The agent loop:
    1. Look at the question and any context gathered so far
    2. Decide: search for more info, or answer with what we have
    3. If searching: generate a search query, retrieve, add to context
    4. Repeat until the agent is confident or max iterations reached
    """
    gathered_context = []
    
    for i in range(max_iterations):
        # Ask the agent what to do next
        context_so_far = "\n\n".join(gathered_context) if gathered_context else "No context gathered yet."
        
        decision_prompt = f"""You are a research assistant. Given a question and context gathered so far, decide your next action.

Question: {question}

Context gathered so far:
{context_so_far}

Respond with EXACTLY one of:
1. SEARCH: <query> - if you need to search for more information (provide a specific search query)
2. ANSWER: <answer> - if you have enough context to answer the question

Your decision:"""
        
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": decision_prompt}],
            max_tokens=256,
            temperature=0
        )
        decision = response.choices[0].message.content.strip()
        
        if verbose:
            print(f"  Iteration {i+1}: {decision[:100]}")
        
        if decision.upper().startswith("ANSWER:"):
            return {
                "answer": decision[7:].strip(),
                "iterations": i + 1,
                "context_pieces": len(gathered_context)
            }
        
        elif decision.upper().startswith("SEARCH:"):
            search_query = decision[7:].strip()
            results = search_knowledge_base(search_query)
            for doc in results:
                if doc not in gathered_context:  # Avoid duplicates
                    gathered_context.append(doc)
            if verbose:
                print(f"           Retrieved {len(results)} passages (total context: {len(gathered_context)})")
        else:
            # If the model didn't follow the format, treat as answer
            return {
                "answer": decision,
                "iterations": i + 1,
                "context_pieces": len(gathered_context)
            }
    
    # Max iterations reached - force an answer
    final_prompt = f"""Based on the following context, answer the question concisely.

Context:
{chr(10).join(gathered_context)}

Question: {question}
Answer:"""
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": final_prompt}],
        max_tokens=256,
        temperature=0
    )
    
    return {
        "answer": response.choices[0].message.content.strip(),
        "iterations": max_iterations,
        "context_pieces": len(gathered_context)
    }

In [16]:
# Test agentic RAG
agent_questions = [
    "What is the speed of light?",
    "Who invented the telephone and when?",
    "What are the main causes of climate change?",
]

for q in agent_questions:
    print(f"\nQ: {q}")
    result = agentic_rag(q, verbose=True)
    print(f"A: {result['answer']}")
    print(f"   ({result['iterations']} iterations, {result['context_pieces']} context pieces)")
    print("-" * 60)


Q: What is the speed of light?
  Iteration 1: SEARCH: "speed of light"
           Retrieved 3 passages (total context: 3)
  Iteration 2: SEARCH: "speed of light" AND "Einstein's relativity" AND "Tesla's criticism"
           Retrieved 3 passages (total context: 5)
  Iteration 3: SEARCH: "speed of light" AND ("Einstein's relativity" OR "Tesla's criticism")
           Retrieved 3 passages (total context: 5)
A: The context does not provide information on the speed of light. It only mentions criticisms from Nikola Tesla towards Albert Einstein's relativity theory.
   (3 iterations, 5 context pieces)
------------------------------------------------------------

Q: Who invented the telephone and when?
  Iteration 1: SEARCH: Alexander Graham Bell
           Retrieved 3 passages (total context: 3)
  Iteration 2: SEARCH: "inventor of telephone" and "Alexander Graham Bell"
           Retrieved 3 passages (total context: 6)
  Iteration 3: SEARCH: "who invented the telephone"
           Retrieved

### Why Agentic RAG Matters

| Aspect | Standard RAG | Agentic RAG |
|--------|-------------|-------------|
| **Retrieval** | Fixed: one query, one search | Adaptive: multiple queries, iterative |
| **Query** | Uses original question as-is | Can reformulate for better retrieval |
| **Failure handling** | No recovery if retrieval fails | Can retry with different strategy |
| **Multi-hop questions** | Struggles | Can decompose and search step by step |
| **Cost** | 1 LLM call | Multiple LLM calls (higher latency) |

**Current industry trend (2025-2026):** Agentic RAG is rapidly becoming the standard for production systems. Frameworks like LangGraph and LlamaIndex Workflows provide structured ways to build these agent loops.

## Summary: The Modern RAG Stack

```
 +------------------+     +------------------+     +------------------+
 |   Data Ingestion |     |    Retrieval     |     |   Generation     |
 |                  |     |                  |     |                  |
 | - Document       | --> | - Vector Store   | --> | - LLM (local or  |
 |   loaders        |     |   (ChromaDB,     |     |   API-based)     |
 | - Text splitters |     |    FAISS, etc.)  |     | - Prompt         |
 | - Embedding      |     | - BM25 + Dense   |     |   templates      |
 |   models         |     | - Reranking      |     | - Agentic loops  |
 +------------------+     +------------------+     +------------------+

 Frameworks: LangChain / LlamaIndex / Haystack / DSPy
```

### What to Choose?

- **Quick prototype**: LlamaIndex (fewest lines of code for basic RAG)
- **Production system**: LangChain + LangGraph (most flexible, largest ecosystem)
- **Research**: DSPy (optimizer-driven, minimal prompt engineering)
- **Enterprise**: Haystack (production-tested, clear contracts)